In [ ]:
# CELL 1
# Install dependencies

!pip install nltk
!pip install sastrawi
!pip install networkx
!pip install scikit-learn
!pip install sentence-transformers
!pip install newspaper3k
!pip install lxml_html_clean

In [ ]:
# CELL 2
# Import libraries

import re
import nltk
import numpy as np
import networkx as nx

from newspaper import Article

from nltk.tokenize import sent_tokenize

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
# CELL 3
# Download tokenizer

nltk.download('punkt')

In [ ]:
# CELL 4
# Load embedding model

embedding_model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

In [ ]:
# CELL 5
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

In [ ]:
# CELL 6
# Input article URL

url = "-"

In [ ]:
# CELL 7
# Download and parse article

article = Article(url, language='id')

article.download()

article.parse()

title = article.title

text = article.text

In [ ]:
# CELL 8
# Show article info

print("TITLE:")
print(title)

print("\nARTICLE:")
print(text[:1500])

In [ ]:
# CELL 9
# Text preprocessing function

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
stop_word_factory = StopWordRemoverFactory()
stopwords = stop_word_factory.create_stop_word_remover()

def preprocess(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-zA-Z\\s]', '', sentence)
    sentence = stopwords.remove(sentence)
    sentence = stemmer.stem(sentence)
    sentence = sentence.strip()
    return sentence

In [ ]:
# CELL 10
# Split article into sentences
import nltk
nltk.download('punkt_tab')

sentences = sent_tokenize(text)

print(f"Total Sentences: {len(sentences)}")

In [ ]:
# CELL 11
# Clean sentences

clean_sentences = [
    preprocess(sentence)
    for sentence in sentences
]

In [ ]:
# CELL 12
# Clean title

clean_title = preprocess(title)

In [ ]:
# CELL 13
# Generate sentence embeddings

sentence_embeddings = embedding_model.encode(
    clean_sentences
)

In [ ]:
# CELL 14
# Build similarity matrix

similarity_matrix = cosine_similarity(
    sentence_embeddings
)

In [ ]:
# CELL 15
# Build TextRank graph

graph = nx.from_numpy_array(
    similarity_matrix
)

scores = nx.pagerank(graph)

In [ ]:
# CELL 16 (HAPUS / ganti menjadi encode title di sini)

# Encode title sekali saja
title_embedding = embedding_model.encode([clean_title])[0]

In [ ]:
# CELL 17
# Position score function

def position_score(index, total_sentences):

    return 1 - (index / total_sentences)

In [ ]:
# CELL 18
# Calculate hybrid scores

hybrid_scores = {}
total_sentences = len(sentences)

for i, sentence in enumerate(clean_sentences):

    textrank_score = scores[i]

    # Gunakan embedding yang sudah ada, bukan encode ulang
    title_score = cosine_similarity(
        [sentence_embeddings[i]],
        [title_embedding]
    )[0][0]

    pos_score = position_score(i, total_sentences)

    final_score = (
        0.6 * textrank_score +
        0.25 * title_score +
        0.15 * pos_score
    )

    hybrid_scores[i] = final_score

In [ ]:
# CELL 19
# Rank sentences

ranked_sentences = sorted(
    (
        (hybrid_scores[i], s, i)
        for i, s in enumerate(sentences)
    ),
    reverse=True
)

In [ ]:
# CELL 20
# Redundancy checker

def is_redundant(sentence_idx, selected_indices, threshold=0.75):

    for selected_idx in selected_indices:

        similarity = cosine_similarity(
            [sentence_embeddings[sentence_idx]],
            [sentence_embeddings[selected_idx]]
        )[0][0]

        if similarity > threshold:
            return True

    return False

In [ ]:
# CELL 21
# Generate dynamic summary

summary_indices = []

compression_rate = 0.3

summary_count = max(
    1,
    int(len(sentences) * compression_rate)
)

for score, sentence, idx in ranked_sentences:

    if not is_redundant(idx, summary_indices):
        summary_indices.append(idx)

    if len(summary_indices) >= summary_count:
        break

# Simpan juga kalimat aslinya untuk cell selanjutnya
summary_sentences = [sentences[i] for i in summary_indices]

In [ ]:
# CELL 22
# Reorder summary

final_summary = []

for sentence in sentences:

    if sentence in summary_sentences:

        final_summary.append(sentence)

summary = " ".join(final_summary)

In [ ]:
# CELL 23
# Show summary

print("TITLE:")
print(title)

print("\nSUMMARY:")
print(summary)

In [ ]:
# CELL 24
# Summary statistics

original_word_count = len(text.split())

summary_word_count = len(summary.split())

compression_ratio = (
    (original_word_count - summary_word_count)
    / original_word_count
) * 100

print("\nSTATISTICS")
print(f"Original Words : {original_word_count}")
print(f"Summary Words  : {summary_word_count}")
print(f"Compression    : {compression_ratio:.2f}%")

In [ ]:
import numpy as np
# Add imports and definitions required by preprocess function for standalone execution
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Re-define stemmer object (originally from CELL 5)
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

# Re-define stopwords object (implicitly used by CELL 9)
stop_word_factory = StopWordRemoverFactory()
stopwords = stop_word_factory.create_stop_word_remover()

# Re-define preprocess function (originally from CELL 9)
def preprocess(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-zA-Z\s]', '', sentence)
    sentence = stopwords.remove(sentence)
    sentence = stemmer.stem(sentence)
    sentence = sentence.strip()
    return sentence

# Preprocess the entire original article text
preprocessed_article_text = preprocess(text)

# Preprocess the generated summary
preprocessed_summary_text = preprocess(summary)

# Encode the preprocessed article and summary
article_embedding = embedding_model.encode([preprocessed_article_text])
summary_embedding = embedding_model.encode([preprocessed_summary_text])

# Calculate cosine similarity
similarity = cosine_similarity(article_embedding, summary_embedding)[0][0]

print(f"Cosine Similarity between Original Article and Summary: {similarity:.4f}")

In [ ]:
# Install the rouge_score library if not already installed
!pip install rouge_score

from rouge_score import rouge_scorer

# Initialize the ROUGE scorer with desired metrics
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Calculate ROUGE scores
# Using the original article as the reference text for ROUGE calculation
# Note: Ideally, ROUGE is compared against a human-written reference summary.
# Using the full article here provides an indication of information recall from the original text.
scores = scorer.score(target=text, prediction=summary)

print("ROUGE Scores:")
for key, value in scores.items():
    print(f"{key}:")
    print(f"  Precision: {value.precision:.4f}")
    print(f"  Recall:    {value.recall:.4f}")
    print(f"  F-measure: {value.fmeasure:.4f}")